# SAFE-Alert — Fold 4 Only (resume)

Chạy lại fold 4 sau khi Kaggle hết giờ ở folds 1-3.

**Setup:** Add dataset `safe-alert-ai-service` → GPU T4 x2 → Run All

In [ ]:
import os, sys, json

AI_SERVICE = '/kaggle/input/safe-alert-ai-service'
WORK_DIR   = '/kaggle/working'

# Auto-detect subfolder nếu zip có thêm 1 level
if not os.path.exists(os.path.join(AI_SERVICE, 'app')):
    for d in os.listdir(AI_SERVICE):
        candidate = os.path.join(AI_SERVICE, d)
        if os.path.isdir(candidate) and os.path.exists(os.path.join(candidate, 'app')):
            AI_SERVICE = candidate
            break

DATA_V2    = os.path.join(AI_SERVICE, 'training_data', 'v2')
PIPELINES  = os.path.join(AI_SERVICE, 'app', 'v2', 'pipelines')
TRAIN_SCRIPT = os.path.join(PIPELINES, 'train_safe_alert.py')
CONFIG       = os.path.join(PIPELINES, 'train_config_research_best.yaml')
ARTIFACT_DIR = os.path.join(WORK_DIR, 'fold4_only')

os.makedirs(ARTIFACT_DIR, exist_ok=True)

for p in [PIPELINES, os.path.join(AI_SERVICE,'app','v2'), AI_SERVICE]:
    sys.path.insert(0, p)

os.chdir(AI_SERVICE)
print(f'AI_SERVICE   : {AI_SERVICE}')
print(f'TRAIN_SCRIPT : {TRAIN_SCRIPT}')
print(f'ARTIFACT_DIR : {ARTIFACT_DIR}')

In [ ]:
!pip install -q ta==0.11.0 vaderSentiment==3.3.2

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
for i in range(torch.cuda.device_count()):
    mem = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)} ({mem:.1f} GB)')

In [ ]:
# Chạy CHỈ fold 4 (--start_fold 4 bỏ qua fold 1-3)
cmd = (
    f'python {TRAIN_SCRIPT}'
    f' --config {CONFIG}'
    f' --walk_forward'
    f' --symbol BTCUSDT'
    f' --horizon 1h'
    f' --epochs 40'
    f' --n_folds 4'
    f' --start_fold 4'
    f' --batch_size 8'
    f' --artifact_dir {ARTIFACT_DIR}'
)
print('Running:', cmd)
os.system(cmd)

In [ ]:
# Đọc kết quả fold 4
m_path = os.path.join(ARTIFACT_DIR, 'fold_4', 'training_metrics.json')
if os.path.exists(m_path):
    with open(m_path) as f:
        m = json.load(f)
    print('=== FOLD 4 RESULT ===')
    meta = m.get('selection_meta', {})
    print(f'  Score   : {meta.get("model_score", 0):.4f}')
    print(f'  Sharpe  : {meta.get("alert_sharpe", 0):.4f}')
    print(f'  PosHit  : {meta.get("position_hit_rate", 0):.4f}')
    print(f'  Deploy  : {m.get("final_deployable", False)}')
    print(f'  Epoch   : {m.get("total_epochs_trained", 0)}')
else:
    print('Chưa có kết quả — kiểm tra lỗi ở cell trên')

In [ ]:
# Zip fold_4 để download
import zipfile, glob

zip_path = os.path.join(WORK_DIR, 'fold4_artifacts.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fpath in glob.glob(os.path.join(ARTIFACT_DIR, '**'), recursive=True):
        if os.path.isfile(fpath):
            zf.write(fpath, os.path.relpath(fpath, WORK_DIR))

print(f'Download: {zip_path} ({os.path.getsize(zip_path)/1e6:.1f} MB)')
print('→ Kaggle Output panel bên phải')